# Notebook 14 — TOP Detection and Tracking

**Scientific objective:** independently establish the TOP-camera detection and tracking baseline while preserving the completed FRONT pipeline as the matched reference.

This notebook validates Roboflow `phys-hus/fish-top-detection` Version 2, audits its YOLO annotations, trains the fixed YOLOv8n baseline, reloads `best.pt` for validation, compares FRONT and TOP under matched detector settings, inventories a real TOP video, and evaluates the ByteTrack B15 transfer baseline when a video is available.

Scope limits: no TOP behavior features, no cross-camera identity matching, no sensor synchronization, no model export, and no claim of official MOT accuracy without identity ground truth. `Front track_id` and `Top track_id` are unrelated camera-local tracker IDs.


## 1. Title and scientific objective
## 2. Provenance


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, json, logging, math, os, platform, re, subprocess, sys, time, warnings
from collections import Counter, defaultdict

import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from IPython.display import display

NOTEBOOK_STARTED_AT = datetime.now(timezone.utc)
NOTEBOOK_START_TIME = time.perf_counter()

def find_project_root(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if all((candidate / marker).exists() for marker in ('AGENTS.md', 'FISH_AI_PROJECT_WORKFLOW.md', '.git')):
            return candidate
    raise RuntimeError(f'Cannot resolve PROJECT_ROOT from {start.resolve()}')

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

PROJECT_ROOT = find_project_root(Path.cwd())
GIT_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, check=True, capture_output=True, text=True).stdout.strip()
CONDA_ENV = os.environ.get('CONDA_DEFAULT_ENV', '')
CUDA_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else 'NOT_AVAILABLE'

print('NOTEBOOK 14 PROVENANCE')
print(f'Datetime UTC: {NOTEBOOK_STARTED_AT.isoformat()}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')
print(f'Conda environment: {CONDA_ENV}')
print(f'Python: {platform.python_version()}')
print(f'Torch: {torch.__version__}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'OpenCV: {cv2.__version__}')
print(f'CUDA available: {CUDA_AVAILABLE}')
print(f'CUDA runtime: {torch.version.cuda}')
print(f'Device: {"cuda:0" if CUDA_AVAILABLE else "cpu"}')
print(f'GPU: {GPU_NAME}')
print(f'Git commit: {GIT_COMMIT}')


## 3. CONFIG

The detector settings exactly match the accepted FRONT YOLOv8n baseline. ByteTrack B15 is a transfer baseline, not an optimized TOP tracker. If there is exactly one TOP video it is selected automatically; when there are multiple videos, set `TOP_VIDEO_RELATIVE` explicitly before **Run All**.


In [ ]:
CAMERA = 'top'
ROBOFLOW_WORKSPACE = 'phys-hus'
ROBOFLOW_PROJECT = 'fish-top-detection'
DATASET_VERSION = 2
EXPORT_FORMAT = 'yolov8'

MODEL_NAME = 'yolov8n.pt'
IMGSZ = 640
EPOCHS = 100
BATCH = 16
DEVICE = 0
SEED = 42
WORKERS = 4
OPTIMIZER = 'auto'
RUNS_PROJECT = PROJECT_ROOT / 'runs' / 'top'
RUN_NAME = 'yolov8n_top_v2_baseline'
RUN_DIR_EXPECTED = RUNS_PROJECT / RUN_NAME

AUDIT_EXPERIMENT_ID = 'TOP_DATASET_AUDIT_V2_001'
TRAIN_EXPERIMENT_ID = 'TOP_DET_YOLOV8N_001'
EVAL_EXPERIMENT_ID = 'TOP_DET_YOLOV8N_EVAL_001'
VIDEO_DET_EXPERIMENT_ID = 'TOP_VIDEO_DET_001'
TRACK_EXPERIMENT_ID = 'TOP_TRACK_BYTETRACK_B15_001'

VIDEO_DETECTION_CONF = 0.68
TRACK_DETECTION_FLOOR = 0.50
NMS_IOU = 0.70
PROGRESS_INTERVAL = 200
TRUE_FISH_COUNT = None
TOP_VIDEO_SPECS = [
    {'video_id': 'TOP_VIDEO_1', 'relative_path': 'data/raw/top/1.mp4'},
    {'video_id': 'TOP_VIDEO_2', 'relative_path': 'data/raw/top/2.mp4'},
]

DATASET_ROOT = PROJECT_ROOT / 'data/roboflow/top_detect_v2'
DATA_YAML = DATASET_ROOT / 'data.yaml'
DATASET_MANIFEST = PROJECT_ROOT / 'results/detection/top_dataset_manifest.json'
TRACKER_CONFIG_PATH = PROJECT_ROOT / 'configs/trackers/top_bytetrack_b15.yaml'
BEST_MODEL = RUN_DIR_EXPECTED / 'weights/best.pt'

CONFIG = {
    'camera': CAMERA, 'workspace': ROBOFLOW_WORKSPACE, 'project': ROBOFLOW_PROJECT,
    'dataset_version': DATASET_VERSION, 'format': EXPORT_FORMAT, 'model': MODEL_NAME,
    'data': str(DATA_YAML.relative_to(PROJECT_ROOT)), 'imgsz': IMGSZ, 'epochs': EPOCHS,
    'batch': BATCH, 'device': DEVICE, 'seed': SEED, 'workers': WORKERS,
    'optimizer': OPTIMIZER, 'project_dir': str(RUNS_PROJECT.relative_to(PROJECT_ROOT)),
    'run_name': RUN_NAME, 'video_detection_conf': VIDEO_DETECTION_CONF,
    'track_detection_floor': TRACK_DETECTION_FLOOR, 'nms_iou': NMS_IOU,
    'tracker_config': str(TRACKER_CONFIG_PATH.relative_to(PROJECT_ROOT)),
    'true_fish_count': TRUE_FISH_COUNT, 'top_videos': TOP_VIDEO_SPECS,
}
print('CONFIG')
print(yaml.safe_dump(CONFIG, sort_keys=False))


## 4. Environment validation
## 5. TOP dataset validation


In [ ]:
assert CONDA_ENV == 'fish', f'FAIL preflight: select the fish kernel, found {CONDA_ENV!r}'
assert DATA_YAML.is_file(), f'FAIL preflight: missing {DATA_YAML}'
assert DATASET_MANIFEST.is_file(), f'FAIL preflight: missing audited manifest {DATASET_MANIFEST}'
assert TRACKER_CONFIG_PATH.is_file(), f'FAIL preflight: missing {TRACKER_CONFIG_PATH}'

with (PROJECT_ROOT / 'configs/data_sources.yaml').open(encoding='utf-8') as handle:
    DATA_SOURCES = yaml.safe_load(handle)
TOP_SOURCE = DATA_SOURCES['sources']['labeled_detection_dataset']['top']
assert TOP_SOURCE == {
    'source': 'roboflow', 'workspace': ROBOFLOW_WORKSPACE, 'project': ROBOFLOW_PROJECT,
    'version': DATASET_VERSION, 'format': EXPORT_FORMAT, 'local_root': 'data/roboflow/top_detect_v2',
}, f'FAIL preflight: unexpected TOP source config: {TOP_SOURCE}'

with DATA_YAML.open(encoding='utf-8') as handle:
    DATA_DEFINITION = yaml.safe_load(handle)
RF_META = DATA_DEFINITION.get('roboflow', {})
assert RF_META.get('workspace') == ROBOFLOW_WORKSPACE
assert RF_META.get('project') == ROBOFLOW_PROJECT
assert int(RF_META.get('version')) == DATASET_VERSION
CLASS_NAMES_RAW = DATA_DEFINITION.get('names')
CLASS_NAMES = {i: name for i, name in enumerate(CLASS_NAMES_RAW)} if isinstance(CLASS_NAMES_RAW, list) else {int(k): v for k, v in CLASS_NAMES_RAW.items()}
assert CLASS_NAMES == {0: 'ca'}, f'FAIL preflight: unexpected TOP classes {CLASS_NAMES}'

EXPECTED_TRACKER = {
    'tracker_type': 'bytetrack', 'track_high_thresh': 0.68, 'track_low_thresh': 0.50,
    'new_track_thresh': 0.68, 'track_buffer': 15, 'match_thresh': 0.80, 'fuse_score': True,
}
TRACKER_CONFIG = yaml.safe_load(TRACKER_CONFIG_PATH.read_text(encoding='utf-8'))
assert TRACKER_CONFIG == EXPECTED_TRACKER, f'FAIL preflight: tracker config differs from fixed transfer baseline: {TRACKER_CONFIG}'

from ultralytics.data.utils import check_det_dataset
ULTRALYTICS_DATA = check_det_dataset(str(DATA_YAML), autodownload=False)
assert Path(ULTRALYTICS_DATA['train']).resolve() == (DATASET_ROOT / 'train/images').resolve()
assert Path(ULTRALYTICS_DATA['val']).resolve() == (DATASET_ROOT / 'valid/images').resolve()
print(f'data.yaml: {DATA_YAML.relative_to(PROJECT_ROOT)}')
print(f'Roboflow metadata: {RF_META}')
print(f'Classes: {CLASS_NAMES}')
print(f'Ultralytics train path: {ULTRALYTICS_DATA["train"]}')
print(f'Ultralytics val path: {ULTRALYTICS_DATA["val"]}')
print(f'Tracker transfer baseline: {TRACKER_CONFIG}')
print('ENVIRONMENT/DATASET PREFLIGHT: PASS')
if not CUDA_AVAILABLE:
    print('WARNING: CUDA is unavailable. The fixed DEVICE=0 training cell will stop rather than silently switch to CPU.')


## 6. Dataset audit
### 7. Class distribution
### 8. Annotation format check
### 9. Duplicate/leakage check

The audit is read-only. It never changes a split or annotation. Exact SHA-256 duplicates and canonical Roboflow source-name collisions are checked across splits; filename-only metadata cannot prove that nearby frames from one recording session are independent.


In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
SPLITS = {'train': DATASET_ROOT / 'train', 'valid': DATASET_ROOT / 'valid'}
if (DATASET_ROOT / 'test').is_dir(): SPLITS['test'] = DATASET_ROOT / 'test'

def canonical_source_key(path):
    stem = re.sub(r'\.rf\.[0-9a-f]+$', '', path.stem, flags=re.IGNORECASE)
    return re.sub(r'_(?:jpg|jpeg|png)$', '', stem, flags=re.IGNORECASE)

split_rows, annotation_rows, class_counter = [], [], Counter()
label_errors, corrupt_images, image_hashes, source_keys = [], [], [], []
resolution_counter = Counter()
for split, root in SPLITS.items():
    images = sorted(p for p in (root / 'images').rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    labels = sorted((root / 'labels').rglob('*.txt'))
    image_map = {p.relative_to(root / 'images').with_suffix(''): p for p in images}
    label_map = {p.relative_to(root / 'labels').with_suffix(''): p for p in labels}
    missing = image_map.keys() - label_map.keys(); orphan = label_map.keys() - image_map.keys()
    empty = bbox_rows = polygon_rows = mixed_files = 0
    for label_path in labels:
        lines = [line for line in label_path.read_text(encoding='utf-8', errors='replace').splitlines() if line.strip()]
        if not lines: empty += 1
        formats = set()
        for line_number, line in enumerate(lines, 1):
            parts = line.split(); row_format = 'bbox' if len(parts) == 5 else ('polygon' if len(parts) >= 7 and len(parts) % 2 == 1 else 'unknown')
            formats.add(row_format)
            reason = None
            try: values = [float(v) for v in parts]
            except ValueError: values, reason = [], 'non_numeric_value'
            if reason is None and not all(math.isfinite(v) for v in values): reason = 'nan_or_inf'
            if reason is None and row_format == 'unknown': reason = 'invalid_field_structure'
            if reason is None and (not values[0].is_integer() or int(values[0]) not in CLASS_NAMES): reason = 'invalid_class_id'
            if reason is None and row_format == 'bbox':
                _, x, y, w, h = values
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1): reason = 'invalid_bbox_coordinates'
                else: bbox_rows += 1; class_counter[(split, int(values[0]))] += 1
            elif reason is None and row_format == 'polygon':
                coords = values[1:]
                if len(coords) < 6 or len(coords) % 2 or not all(0 <= v <= 1 for v in coords): reason = 'invalid_polygon_coordinates'
                else: polygon_rows += 1; class_counter[(split, int(values[0]))] += 1
            if reason: label_errors.append({'split': split, 'label_path': str(label_path.relative_to(DATASET_ROOT)), 'line_number': line_number, 'error': reason})
        if 'bbox' in formats and 'polygon' in formats: mixed_files += 1
    for image_path in images:
        image = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)
        relative = str(image_path.relative_to(DATASET_ROOT))
        if image is None: corrupt_images.append({'split': split, 'image_path': relative})
        else:
            height, width = image.shape[:2]; channels = 1 if image.ndim == 2 else image.shape[2]
            resolution_counter[(split, width, height, channels)] += 1
        image_hashes.append({'split': split, 'path': relative, 'sha256': sha256_file(image_path)})
        source_keys.append({'split': split, 'path': relative, 'source_key': canonical_source_key(image_path)})
    split_rows.append({'split': split, 'n_images': len(images), 'n_label_files': len(labels), 'n_missing_labels': len(missing), 'n_orphan_labels': len(orphan), 'n_empty_labels': empty, 'n_boxes': bbox_rows, 'bbox_rows': bbox_rows, 'polygon_rows': polygon_rows, 'mixed_format_files': mixed_files})
    annotation_rows.append({'split': split, 'bbox_rows': bbox_rows, 'polygon_rows': polygon_rows, 'mixed_format_files': mixed_files})

hash_df = pd.DataFrame(image_hashes)
duplicate_groups = []
for digest, group in hash_df.groupby('sha256'):
    if len(group) > 1:
        splits = sorted(group['split'].unique())
        duplicate_groups.append({'sha256': digest, 'file_count': len(group), 'splits': '|'.join(splits), 'cross_split': len(splits) > 1, 'paths': '|'.join(group['path'])})
source_df = pd.DataFrame(source_keys)
source_cross_split = []
for key, group in source_df.groupby('source_key'):
    splits = sorted(group['split'].unique())
    if len(splits) > 1: source_cross_split.append({'source_key': key, 'file_count': len(group), 'splits': '|'.join(splits), 'paths': '|'.join(group['path'])})

AUDIT_DF = pd.DataFrame(split_rows)
CLASS_DF = pd.DataFrame([{'class_id': class_id, 'class_name': name, **{f'{split}_boxes': class_counter[(split, class_id)] for split in ('train', 'valid', 'test')}} for class_id, name in CLASS_NAMES.items()])
CLASS_DF['total_boxes'] = CLASS_DF[['train_boxes', 'valid_boxes', 'test_boxes']].sum(axis=1)
ANNOTATION_DF = pd.DataFrame(annotation_rows)
RESOLUTION_DF = pd.DataFrame([{'split': split, 'width': width, 'height': height, 'channels': channels, 'count': count} for (split, width, height, channels), count in sorted(resolution_counter.items())])
DUPLICATE_DF = pd.DataFrame(duplicate_groups, columns=['sha256', 'file_count', 'splits', 'cross_split', 'paths'])
LEAKAGE_DF = pd.DataFrame(source_cross_split, columns=['source_key', 'file_count', 'splits', 'paths'])

AUDIT_WARNINGS = []
if 'test' not in SPLITS: AUDIT_WARNINGS.append('Optional test split is absent; the Roboflow YAML test declaration has no local test directory.')
if not any(re.search(r'(?:^|[_-])T[12](?:[_-]|$)', row['source_key'], flags=re.IGNORECASE) for row in source_keys): AUDIT_WARNINGS.append('T1/T2 stratification unavailable from exported filenames/metadata.')
if CLASS_NAMES == {0: 'ca'}: AUDIT_WARNINGS.append("TOP class 0 name 'ca' differs by letter case from FRONT reference 'Ca'; class ID is compatible and labels are preserved unchanged.")
AUDIT_WARNINGS.append('Generic t_<time>_<index> filenames lack session identifiers; nearby-frame leakage cannot be excluded from export metadata alone.')

fatal_counts = {'missing_labels': int(AUDIT_DF.n_missing_labels.sum()), 'orphan_labels': int(AUDIT_DF.n_orphan_labels.sum()), 'invalid_rows': len(label_errors), 'corrupt_images': len(corrupt_images)}
assert not any(fatal_counts.values()), f'FAIL dataset audit: {fatal_counts}'
assert int(ANNOTATION_DF.polygon_rows.sum()) == 0, 'FAIL dataset audit: polygon rows require review before detection training.'
assert int(ANNOTATION_DF.mixed_format_files.sum()) == 0, 'FAIL dataset audit: mixed annotation files require review.'
assert not len(DUPLICATE_DF[DUPLICATE_DF.cross_split == True]), 'FAIL dataset audit: exact cross-split duplicate hashes.'

RESULTS_DET = PROJECT_ROOT / 'results/detection'; AUDIT_LOG_DIR = PROJECT_ROOT / 'logs/detection' / AUDIT_EXPERIMENT_ID
RESULTS_DET.mkdir(parents=True, exist_ok=True); AUDIT_LOG_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DF.to_csv(RESULTS_DET / 'top_dataset_audit_summary.csv', index=False)
CLASS_DF.to_csv(RESULTS_DET / 'top_class_distribution.csv', index=False)
ANNOTATION_DF.to_csv(RESULTS_DET / 'top_annotation_format_summary.csv', index=False)
RESOLUTION_DF.to_csv(RESULTS_DET / 'top_resolution_distribution.csv', index=False)
DUPLICATE_DF.to_csv(RESULTS_DET / 'top_duplicate_files.csv', index=False)
LEAKAGE_DF.to_csv(RESULTS_DET / 'top_possible_split_leakage.csv', index=False)
AUDIT_SUMMARY = {
    'experiment_id': AUDIT_EXPERIMENT_ID, 'workspace': ROBOFLOW_WORKSPACE, 'project': ROBOFLOW_PROJECT,
    'version': DATASET_VERSION, 'format': EXPORT_FORMAT, 'dataset_root': str(DATASET_ROOT.relative_to(PROJECT_ROOT)),
    'split_counts': AUDIT_DF.to_dict(orient='records'), 'classes': CLASS_NAMES,
    'invalid_label_rows': len(label_errors), 'corrupt_images': len(corrupt_images),
    'exact_duplicate_groups': len(DUPLICATE_DF), 'canonical_source_cross_split_groups': len(LEAKAGE_DF),
    'checkpoint_result': 'PASS_WITH_WARNING' if AUDIT_WARNINGS else 'PASS', 'warnings': AUDIT_WARNINGS,
}
(AUDIT_LOG_DIR / 'summary.json').write_text(json.dumps(AUDIT_SUMMARY, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
display(AUDIT_DF); display(CLASS_DF); display(ANNOTATION_DF); display(RESOLUTION_DF)
print(f'Exact duplicate groups: {len(DUPLICATE_DF)}; canonical source cross-split groups: {len(LEAKAGE_DF)}')
print(f'AUDIT RESULT: {AUDIT_SUMMARY["checkpoint_result"]}')
for warning in AUDIT_WARNINGS: print(f'WARNING: {warning}')


## 10. YOLOv8n TOP baseline

This is the fixed matched baseline `TOP_DET_YOLOV8N_001`. The cell intentionally requires CUDA because `DEVICE=0`; it will not silently change the approved baseline to CPU. Existing run output is preserved and causes a preflight stop.


In [ ]:
assert CUDA_AVAILABLE, 'FAIL training preflight: CUDA is required because the approved baseline fixes DEVICE=0.'
if RUN_DIR_EXPECTED.exists() and any(RUN_DIR_EXPECTED.iterdir()):
    raise RuntimeError(f'FAIL training preflight: preserve existing run output: {RUN_DIR_EXPECTED.relative_to(PROJECT_ROOT)}')

from ultralytics import YOLO
TRAIN_MODEL = YOLO(MODEL_NAME)
TRAIN_START = time.perf_counter()
TRAIN_RESULTS = TRAIN_MODEL.train(
    data=str(DATA_YAML), model=MODEL_NAME, imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH,
    device=DEVICE, seed=SEED, workers=WORKERS, optimizer=OPTIMIZER, deterministic=True,
    project=str(RUNS_PROJECT), name=RUN_NAME, exist_ok=False, verbose=True, plots=True, val=True,
)
TRAINING_RUNTIME_SEC = time.perf_counter() - TRAIN_START
RUN_DIR = Path(TRAIN_MODEL.trainer.save_dir).resolve()
BEST_MODEL = RUN_DIR / 'weights/best.pt'; TRAIN_RESULTS_CSV = RUN_DIR / 'results.csv'
assert BEST_MODEL.is_file() and TRAIN_RESULTS_CSV.is_file(), 'FAIL: training did not produce best.pt/results.csv.'
BEST_MODEL_SHA256 = sha256_file(BEST_MODEL)
history = pd.read_csv(TRAIN_RESULTS_CSV); history.columns = [column.strip() for column in history.columns]
best_index = history['metrics/mAP50-95(B)'].astype(float).idxmax(); best_row = history.loc[best_index]
TRAIN_METRICS = {
    'experiment_id': TRAIN_EXPERIMENT_ID, 'best_epoch': int(best_row['epoch']) + 1,
    'precision': float(best_row['metrics/precision(B)']), 'recall': float(best_row['metrics/recall(B)']),
    'mAP50': float(best_row['metrics/mAP50(B)']), 'mAP50_95': float(best_row['metrics/mAP50-95(B)']),
    'training_runtime_sec': round(TRAINING_RUNTIME_SEC, 3), 'best_model_sha256': BEST_MODEL_SHA256,
    'dataset_version': DATASET_VERSION, 'git_commit': GIT_COMMIT,
}
TRAIN_LOG_DIR = PROJECT_ROOT / 'logs/detection' / TRAIN_EXPERIMENT_ID; TRAIN_LOG_DIR.mkdir(parents=True, exist_ok=True)
(TRAIN_LOG_DIR / 'config.yaml').write_text(yaml.safe_dump({**CONFIG, 'experiment_id': TRAIN_EXPERIMENT_ID}, sort_keys=False), encoding='utf-8')
(TRAIN_LOG_DIR / 'environment.txt').write_text('\n'.join([f'python={platform.python_version()}', f'python_executable={sys.executable}', f'conda_env={CONDA_ENV}', f'torch={torch.__version__}', f'cuda_available={CUDA_AVAILABLE}', f'gpu={GPU_NAME}', f'ultralytics={ultralytics.__version__}', f'git_commit={GIT_COMMIT}']) + '\n', encoding='utf-8')
(TRAIN_LOG_DIR / 'summary.json').write_text(json.dumps({**TRAIN_METRICS, 'model': MODEL_NAME, 'data': str(DATA_YAML.relative_to(PROJECT_ROOT)), 'best_model': str(BEST_MODEL.relative_to(PROJECT_ROOT)), 'checkpoint_result': 'PASS', 'warnings': []}, indent=2) + '\n', encoding='utf-8')
pd.DataFrame([TRAIN_METRICS]).to_csv(RESULTS_DET / 'top_yolov8n_baseline_metrics.csv', index=False)
print(f'Training completed: {TRAINING_RUNTIME_SEC:.1f}s; best epoch={TRAIN_METRICS["best_epoch"]}; best.pt SHA-256={BEST_MODEL_SHA256}')


## 11. Reloaded best-model validation
## 12. FRONT vs TOP comparison

Validation metrics are reported on TOP `valid`, not as test accuracy. The FRONT row uses its accepted reloaded-best-model validation evidence.


In [ ]:
EVAL_LOG_DIR = PROJECT_ROOT / 'logs/detection' / EVAL_EXPERIMENT_ID
EVAL_OUTPUT_ROOT = PROJECT_ROOT / 'outputs/top/detection/evaluation'
if (EVAL_OUTPUT_ROOT / EVAL_EXPERIMENT_ID).exists():
    raise RuntimeError(f'FAIL evaluation preflight: preserve existing output {EVAL_OUTPUT_ROOT / EVAL_EXPERIMENT_ID}')

EVAL_MODEL = YOLO(str(BEST_MODEL), task='detect')
MODEL_INFO = EVAL_MODEL.info(verbose=True, imgsz=IMGSZ)
MODEL_PARAMETERS = int(sum(parameter.numel() for parameter in EVAL_MODEL.model.parameters()))
MODEL_GFLOPS = float(MODEL_INFO[-1]) if isinstance(MODEL_INFO, tuple) and MODEL_INFO and isinstance(MODEL_INFO[-1], (int, float)) else None
VAL_START = time.perf_counter()
VAL_RESULTS = EVAL_MODEL.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ, device=DEVICE, plots=True, project=str(EVAL_OUTPUT_ROOT), name=EVAL_EXPERIMENT_ID, exist_ok=False, verbose=True)
VALIDATION_RUNTIME_SEC = time.perf_counter() - VAL_START
speed = {key: float(value) for key, value in VAL_RESULTS.speed.items()}
TOP_VAL_METRICS = {
    'scope': 'overall', 'class_id': None, 'class_name': 'all', 'precision': float(VAL_RESULTS.box.mp),
    'recall': float(VAL_RESULTS.box.mr), 'mAP50': float(VAL_RESULTS.box.map50), 'mAP50_95': float(VAL_RESULTS.box.map),
    'parameters': MODEL_PARAMETERS, 'GFLOPs': MODEL_GFLOPS, 'model_size_MB': BEST_MODEL.stat().st_size / (1024 ** 2),
    'preprocess_ms': speed.get('preprocess'), 'inference_ms': speed.get('inference'), 'postprocess_ms': speed.get('postprocess'),
    'validation_runtime_sec': round(VALIDATION_RUNTIME_SEC, 3), 'validation_images': int(AUDIT_DF.loc[AUDIT_DF.split == 'valid', 'n_images'].iloc[0]),
    'validation_objects': int(AUDIT_DF.loc[AUDIT_DF.split == 'valid', 'n_boxes'].iloc[0]),
}
pd.DataFrame([TOP_VAL_METRICS]).to_csv(RESULTS_DET / 'top_yolov8n_validation_metrics.csv', index=False)
repro_rows = []
for metric in ('precision', 'recall', 'mAP50', 'mAP50_95'):
    repro_rows.append({'metric': metric, 'training_validation': TRAIN_METRICS[metric], 'reloaded_best_model_validation': TOP_VAL_METRICS[metric], 'absolute_difference': abs(TRAIN_METRICS[metric] - TOP_VAL_METRICS[metric])})
pd.DataFrame(repro_rows).to_csv(RESULTS_DET / 'top_yolov8n_reproducibility_check.csv', index=False)

front_metrics_path = PROJECT_ROOT / 'results/detection/front_yolov8n_validation_metrics.csv'
front_overall = pd.read_csv(front_metrics_path).query("scope == 'overall'").iloc[0]
comparison_rows = [
    {'camera': 'front', 'dataset_version': 1, 'model': 'yolov8n.pt', 'imgsz': 640, 'epochs': 100, 'batch': 16, 'precision': float(front_overall.precision), 'recall': float(front_overall.recall), 'mAP50': float(front_overall.mAP50), 'mAP50_95': float(front_overall.mAP50_95), 'n_validation_images': 254, 'n_validation_objects': 995, 'parameters': 3011043, 'GFLOPs': 8.1917, 'model_size_MB': 5.95, 'inference_ms': 4.00},
    {'camera': 'top', 'dataset_version': 2, 'model': MODEL_NAME, 'imgsz': IMGSZ, 'epochs': EPOCHS, 'batch': BATCH, 'precision': TOP_VAL_METRICS['precision'], 'recall': TOP_VAL_METRICS['recall'], 'mAP50': TOP_VAL_METRICS['mAP50'], 'mAP50_95': TOP_VAL_METRICS['mAP50_95'], 'n_validation_images': TOP_VAL_METRICS['validation_images'], 'n_validation_objects': TOP_VAL_METRICS['validation_objects'], 'parameters': MODEL_PARAMETERS, 'GFLOPs': MODEL_GFLOPS, 'model_size_MB': TOP_VAL_METRICS['model_size_MB'], 'inference_ms': TOP_VAL_METRICS['inference_ms']},
]
COMPARISON_DF = pd.DataFrame(comparison_rows); COMPARISON_DF.to_csv(RESULTS_DET / 'front_top_detection_comparison.csv', index=False)
EVAL_LOG_DIR.mkdir(parents=True, exist_ok=True)
(EVAL_LOG_DIR / 'config.yaml').write_text(yaml.safe_dump({'experiment_id': EVAL_EXPERIMENT_ID, 'model': str(BEST_MODEL.relative_to(PROJECT_ROOT)), 'model_sha256': BEST_MODEL_SHA256, 'data': str(DATA_YAML.relative_to(PROJECT_ROOT)), 'split': 'val', 'imgsz': IMGSZ, 'device': DEVICE}, sort_keys=False), encoding='utf-8')
(EVAL_LOG_DIR / 'environment.txt').write_text((TRAIN_LOG_DIR / 'environment.txt').read_text(encoding='utf-8'), encoding='utf-8')
(EVAL_LOG_DIR / 'summary.json').write_text(json.dumps({'experiment_id': EVAL_EXPERIMENT_ID, **TOP_VAL_METRICS, 'model_sha256': BEST_MODEL_SHA256, 'checkpoint_result': 'PASS', 'warnings': [], 'note': 'Validation metrics are not test accuracy.'}, indent=2) + '\n', encoding='utf-8')
display(pd.DataFrame([TOP_VAL_METRICS])); display(COMPARISON_DF)
print('Do not declare one camera better from a single metric; inspect dataset and failure modes.')


## 13. TOP video discovery — two independent sequences

This section is deliberately self-contained: after the accepted detector has been trained and validated, the USER may restart the kernel and run from this heading downward. It loads the existing `best.pt`; it never calls training or validation.

`TOP_VIDEO_1` and `TOP_VIDEO_2` are separate tracker sequences. A fresh YOLO/ByteTrack object, tracker state, trajectory-tail buffer, frame index, and local track-ID namespace are created for each video. A numeric ID in one video has no relation to the same numeric ID in the other video.


In [14]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, subprocess, sys, time
from collections import defaultdict, deque

import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from IPython.display import display

VIDEO_SECTION_STARTED_AT = datetime.now(timezone.utc)
VIDEO_SECTION_START_TIME = time.perf_counter()

def find_project_root_for_video(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if all((candidate / marker).exists() for marker in ('AGENTS.md', 'FISH_AI_PROJECT_WORKFLOW.md', '.git')):
            return candidate
    raise RuntimeError(f'Cannot resolve PROJECT_ROOT from {start.resolve()}')

def sha256_video_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

PROJECT_ROOT = find_project_root_for_video(Path.cwd())
CONDA_ENV = os.environ.get('CONDA_DEFAULT_ENV', '')
GIT_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, check=True, capture_output=True, text=True).stdout.strip()
CUDA_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else 'NOT_AVAILABLE'

MODEL_NAME = 'yolov8n.pt'
IMGSZ = 640
DEVICE = 0
VIDEO_DETECTION_CONF = 0.68
TRACK_DETECTION_FLOOR = 0.50
NMS_IOU = 0.70
PROGRESS_INTERVAL = 200
TRUE_FISH_COUNT = None
VIDEO_DET_EXPERIMENT_ID = 'TOP_VIDEO_DET_001'
TRACK_EXPERIMENT_ID = 'TOP_TRACK_BYTETRACK_B15_001'
TOP_VIDEO_SPECS = [
    {'video_id': 'TOP_VIDEO_1', 'relative_path': 'data/raw/top/1.mp4'},
    {'video_id': 'TOP_VIDEO_2', 'relative_path': 'data/raw/top/2.mp4'},
]

BEST_MODEL = PROJECT_ROOT / 'runs/top/yolov8n_top_v2_baseline/weights/best.pt'
TRAIN_METRICS_PATH = PROJECT_ROOT / 'results/detection/top_yolov8n_baseline_metrics.csv'
VALIDATION_METRICS_PATH = PROJECT_ROOT / 'results/detection/top_yolov8n_validation_metrics.csv'
TRACKER_CONFIG_PATH = PROJECT_ROOT / 'configs/trackers/top_bytetrack_b15.yaml'
assert CONDA_ENV == 'fish', f'FAIL video preflight: select fish kernel, found {CONDA_ENV!r}'
assert BEST_MODEL.is_file(), f'FAIL video preflight: missing accepted model {BEST_MODEL}'
assert TRAIN_METRICS_PATH.is_file() and VALIDATION_METRICS_PATH.is_file(), 'FAIL video preflight: accepted detector evidence is missing.'
assert TRACKER_CONFIG_PATH.is_file(), f'FAIL video preflight: missing {TRACKER_CONFIG_PATH}'

ACCEPTED_TRAIN_METRICS = pd.read_csv(TRAIN_METRICS_PATH).iloc[0].to_dict()
ACCEPTED_VALIDATION_METRICS = pd.read_csv(VALIDATION_METRICS_PATH).query("scope == 'overall'").iloc[0].to_dict()
BEST_MODEL_SHA256 = sha256_video_file(BEST_MODEL)
assert BEST_MODEL_SHA256 == str(ACCEPTED_TRAIN_METRICS['best_model_sha256']), 'FAIL video preflight: best.pt SHA-256 differs from accepted training evidence.'

EXPECTED_TRACKER = {
    'tracker_type': 'bytetrack', 'track_high_thresh': 0.68, 'track_low_thresh': 0.50,
    'new_track_thresh': 0.68, 'track_buffer': 15, 'match_thresh': 0.80, 'fuse_score': True,
}
TRACKER_CONFIG = yaml.safe_load(TRACKER_CONFIG_PATH.read_text(encoding='utf-8'))
assert TRACKER_CONFIG == EXPECTED_TRACKER, f'FAIL video preflight: unexpected tracker config {TRACKER_CONFIG}'

TOP_VIDEO_CONTEXTS = []
for spec in TOP_VIDEO_SPECS:
    path = (PROJECT_ROOT / spec['relative_path']).resolve()
    assert path.is_file(), f'FAIL video preflight: missing {spec["relative_path"]}'
    capture = cv2.VideoCapture(str(path))
    assert capture.isOpened(), f'FAIL video preflight: cannot open {spec["relative_path"]}'
    fps = float(capture.get(cv2.CAP_PROP_FPS))
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    ok, first_frame = capture.read()
    capture.release()
    assert ok and first_frame is not None, f'FAIL video preflight: cannot decode first frame of {spec["relative_path"]}'
    assert fps > 0 and frame_count > 0 and width > 0 and height > 0
    assert first_frame.shape[1] == width and first_frame.shape[0] == height
    TOP_VIDEO_CONTEXTS.append({
        **spec, 'path': path, 'source_video': path.name, 'fps': fps, 'frame_count': frame_count,
        'width': width, 'height': height, 'duration_sec': frame_count / fps,
        'video_sha256': sha256_video_file(path),
    })

TOP_VIDEO_INVENTORY_DF = pd.DataFrame([{key: value for key, value in context.items() if key not in ('path',)} for context in TOP_VIDEO_CONTEXTS])
print('VIDEO/TRACKING CONFIG')
print(yaml.safe_dump({
    'model': str(BEST_MODEL.relative_to(PROJECT_ROOT)), 'model_sha256': BEST_MODEL_SHA256,
    'imgsz': IMGSZ, 'device': DEVICE, 'video_detection_conf': VIDEO_DETECTION_CONF,
    'tracking_detection_floor': TRACK_DETECTION_FLOOR, 'iou': NMS_IOU,
    'tracker': str(TRACKER_CONFIG_PATH.relative_to(PROJECT_ROOT)), 'videos': TOP_VIDEO_SPECS,
    'true_fish_count': TRUE_FISH_COUNT,
}, sort_keys=False))
display(TOP_VIDEO_INVENTORY_DF)
print(f'Accepted detector best epoch: {int(ACCEPTED_TRAIN_METRICS["best_epoch"])}')
print(f'Accepted TOP validation: P={ACCEPTED_VALIDATION_METRICS["precision"]:.6f}, R={ACCEPTED_VALIDATION_METRICS["recall"]:.6f}, mAP50={ACCEPTED_VALIDATION_METRICS["mAP50"]:.6f}, mAP50-95={ACCEPTED_VALIDATION_METRICS["mAP50_95"]:.6f}')
print(f'CUDA available: {CUDA_AVAILABLE}; GPU: {GPU_NAME}')
print('VIDEO INPUT PREFLIGHT: PASS — two independent videos; no detector training or validation will run below.')


VIDEO/TRACKING CONFIG
model: runs/top/yolov8n_top_v2_baseline/weights/best.pt
model_sha256: 216174c3a40d57bfd7b0d7e46eec90974ddb0b27c879dd1961543e3233a6c5e5
imgsz: 640
device: 0
video_detection_conf: 0.68
tracking_detection_floor: 0.5
iou: 0.7
tracker: configs/trackers/top_bytetrack_b15.yaml
videos:
- video_id: TOP_VIDEO_1
  relative_path: data/raw/top/1.mp4
- video_id: TOP_VIDEO_2
  relative_path: data/raw/top/2.mp4
true_fish_count: null



,video_id,relative_path,source_video,fps,frame_count,width,height,duration_sec,video_sha256
0,TOP_VIDEO_1,data/raw/top/1.mp4,1.mp4,19.350643,1119,1280,960,57.827536,57d09e2c4613fc383c2532d36f53bf15866409673ba274...
1,TOP_VIDEO_2,data/raw/top/2.mp4,2.mp4,17.172190,2031,1280,960,118.272623,a92642426eb4a10d4390613a540cdc5a5374e22656b05e...


Accepted detector best epoch: 65
Accepted TOP validation: P=0.967531, R=0.971339, mAP50=0.983417, mAP50-95=0.606223
CUDA available: True; GPU: NVIDIA GeForce RTX 3050
VIDEO INPUT PREFLIGHT: PASS — two independent videos; no detector training or validation will run below.


## 14. TOP video detection diagnostics — per video

The accepted TOP detector is loaded directly from `runs/top/yolov8n_top_v2_baseline/weights/best.pt`. Detection is run independently on `1.mp4` and `2.mp4` using the fixed `imgsz`, confidence, and IoU settings. No detector or tracker parameter is tuned.

These are operational count diagnostics, not bbox Precision/Recall/mAP. Because the true fish count is not established by cited ground truth, exact-count accuracy, MAE, and count bias are intentionally omitted.


In [15]:
from ultralytics import YOLO

assert CUDA_AVAILABLE, 'FAIL video detection preflight: fixed DEVICE=0 requires CUDA; do not silently change the accepted configuration.'
DETECTION_OUTPUT_DIR = PROJECT_ROOT / 'outputs/top/detection'
DETECTION_RESULTS_PATH = PROJECT_ROOT / 'results/detection/top_video_detection_summary.csv'
DETECTION_LOG_DIR = PROJECT_ROOT / 'logs/detection' / VIDEO_DET_EXPERIMENT_ID
expected_detection_outputs = []
for context in TOP_VIDEO_CONTEXTS:
    stem = Path(context['source_video']).stem
    expected_detection_outputs.extend([
        DETECTION_OUTPUT_DIR / f'{stem}_detection_per_frame.csv',
        DETECTION_OUTPUT_DIR / f'{stem}_detection_overlay.mp4',
    ])
existing_detection_outputs = [path for path in [*expected_detection_outputs, DETECTION_RESULTS_PATH] if path.exists()]
if DETECTION_LOG_DIR.exists() and any(DETECTION_LOG_DIR.iterdir()): existing_detection_outputs.append(DETECTION_LOG_DIR)
if existing_detection_outputs:
    raise RuntimeError(f'FAIL video detection preflight: preserve existing outputs before rerun: {[str(path.relative_to(PROJECT_ROOT)) for path in existing_detection_outputs]}')

DETECTION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DETECTION_LOG_DIR.mkdir(parents=True, exist_ok=False)
VIDEO_DETECTION_SUMMARIES = []
DETECTION_PER_FRAME = {}
for context in TOP_VIDEO_CONTEXTS:
    source_video = context['source_video']; stem = Path(source_video).stem
    per_frame_path = DETECTION_OUTPUT_DIR / f'{stem}_detection_per_frame.csv'
    overlay_path = DETECTION_OUTPUT_DIR / f'{stem}_detection_overlay.mp4'
    model = YOLO(str(BEST_MODEL), task='detect')
    capture = cv2.VideoCapture(str(context['path']))
    writer = cv2.VideoWriter(str(overlay_path), cv2.VideoWriter_fourcc(*'mp4v'), context['fps'], (context['width'], context['height']))
    assert capture.isOpened() and writer.isOpened(), f'FAIL detection output setup for {source_video}'
    rows = []; frame_index = 0; started = time.perf_counter()
    print(f'DETECTION START {source_video}: frames={context["frame_count"]}, fps={context["fps"]:.6f}, resolution={context["width"]}x{context["height"]}')
    try:
        while True:
            ok, frame = capture.read()
            if not ok: break
            result = model.predict(frame, conf=VIDEO_DETECTION_CONF, iou=NMS_IOU, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
            count = len(result.boxes) if result.boxes is not None else 0
            rows.append({'source_video': source_video, 'frame_index': frame_index, 'time_sec': frame_index / context['fps'], 'detections': count})
            writer.write(result.plot())
            frame_index += 1
            if frame_index % PROGRESS_INTERVAL == 0 or frame_index == context['frame_count']:
                elapsed = time.perf_counter() - started
                print(f'{source_video}: {frame_index}/{context["frame_count"]} | elapsed={elapsed:.1f}s | processing FPS={frame_index/elapsed:.2f}')
    finally:
        capture.release(); writer.release()
    assert frame_index == context['frame_count'], f'FAIL detection: incomplete {source_video}: {frame_index}/{context["frame_count"]}'
    runtime_sec = time.perf_counter() - started
    frame_df = pd.DataFrame(rows); frame_df.to_csv(per_frame_path, index=False)
    DETECTION_PER_FRAME[source_video] = frame_df
    VIDEO_DETECTION_SUMMARIES.append({
        'video': source_video, 'video_id': context['video_id'], 'fps': context['fps'],
        'frame_count': context['frame_count'], 'duration_sec': context['duration_sec'],
        'mean_detections_per_frame': float(frame_df.detections.mean()),
        'median_detections_per_frame': float(frame_df.detections.median()),
        'min_detections': int(frame_df.detections.min()), 'max_detections': int(frame_df.detections.max()),
        'processing_fps': context['frame_count'] / runtime_sec, 'true_fish_count': None,
        'per_frame_output': str(per_frame_path.relative_to(PROJECT_ROOT)),
        'overlay_output': str(overlay_path.relative_to(PROJECT_ROOT)),
    })
    print(f'Created {per_frame_path.relative_to(PROJECT_ROOT)} ({len(frame_df)} frames)')
    print(f'Created {overlay_path.relative_to(PROJECT_ROOT)} ({overlay_path.stat().st_size} bytes)')

VIDEO_DETECTION_SUMMARY_DF = pd.DataFrame(VIDEO_DETECTION_SUMMARIES)
DETECTION_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
VIDEO_DETECTION_SUMMARY_DF.to_csv(DETECTION_RESULTS_PATH, index=False)
(DETECTION_LOG_DIR / 'config.yaml').write_text(yaml.safe_dump({
    'experiment_id': VIDEO_DET_EXPERIMENT_ID, 'model': str(BEST_MODEL.relative_to(PROJECT_ROOT)),
    'model_sha256': BEST_MODEL_SHA256, 'imgsz': IMGSZ, 'conf': VIDEO_DETECTION_CONF,
    'iou': NMS_IOU, 'device': DEVICE, 'videos': TOP_VIDEO_SPECS,
}, sort_keys=False), encoding='utf-8')
(DETECTION_LOG_DIR / 'environment.txt').write_text('\n'.join([
    f'datetime_utc={VIDEO_SECTION_STARTED_AT.isoformat()}', f'git_commit={GIT_COMMIT}',
    f'python_executable={sys.executable}', f'python={platform.python_version()}', f'conda_env={CONDA_ENV}',
    f'torch={torch.__version__}', f'cuda_available={CUDA_AVAILABLE}', f'gpu={GPU_NAME}',
    f'ultralytics={ultralytics.__version__}', f'opencv={cv2.__version__}',
]) + '\n', encoding='utf-8')
(DETECTION_LOG_DIR / 'summary.json').write_text(json.dumps({
    'experiment_id': VIDEO_DET_EXPERIMENT_ID, 'model_sha256': BEST_MODEL_SHA256,
    'per_video': {row['video']: row for row in VIDEO_DETECTION_SUMMARIES},
    'metric_scope': 'operational detection count diagnostics; true fish count unavailable',
    'checkpoint_result': 'PASS_WITH_WARNING',
    'warnings': ['True fish count has not been established; no exact-count accuracy or count-error metric is reported.'],
}, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
display(VIDEO_DETECTION_SUMMARY_DF)
print(f'Created {DETECTION_RESULTS_PATH.relative_to(PROJECT_ROOT)} ({len(VIDEO_DETECTION_SUMMARY_DF)} rows)')


DETECTION START 1.mp4: frames=1119, fps=19.350643, resolution=1280x960
1.mp4: 200/1119 | elapsed=4.3s | processing FPS=46.96
1.mp4: 400/1119 | elapsed=8.3s | processing FPS=48.01
1.mp4: 600/1119 | elapsed=12.3s | processing FPS=48.83
1.mp4: 800/1119 | elapsed=16.3s | processing FPS=49.01
1.mp4: 1000/1119 | elapsed=20.2s | processing FPS=49.42
1.mp4: 1119/1119 | elapsed=22.7s | processing FPS=49.40
Created outputs/top/detection/1_detection_per_frame.csv (1119 frames)
Created outputs/top/detection/1_detection_overlay.mp4 (32605633 bytes)
DETECTION START 2.mp4: frames=2031, fps=17.172190, resolution=1280x960
2.mp4: 200/2031 | elapsed=4.0s | processing FPS=49.40
2.mp4: 400/2031 | elapsed=7.9s | processing FPS=50.85
2.mp4: 600/2031 | elapsed=11.8s | processing FPS=50.89
2.mp4: 800/2031 | elapsed=15.6s | processing FPS=51.31
2.mp4: 1000/2031 | elapsed=19.4s | processing FPS=51.56
2.mp4: 1200/2031 | elapsed=23.3s | processing FPS=51.59
2.mp4: 1400/2031 | elapsed=27.1s | processing FPS=51.73
2

,video,video_id,fps,frame_count,duration_sec,mean_detections_per_frame,median_detections_per_frame,min_detections,max_detections,processing_fps,true_fish_count,per_frame_output,overlay_output
0,1.mp4,TOP_VIDEO_1,19.350643,1119,57.827536,1.887399,2.0,0,5,49.394147,None,outputs/top/detection/1_detection_per_frame.csv,outputs/top/detection/1_detection_overlay.mp4
1,2.mp4,TOP_VIDEO_2,17.172190,2031,118.272623,2.910881,3.0,0,6,51.508676,None,outputs/top/detection/2_detection_per_frame.csv,outputs/top/detection/2_detection_overlay.mp4


Created results/detection/top_video_detection_summary.csv (2 rows)


## 15. ByteTrack B15 transfer baseline — reset per video
## 16. Tracking, trajectory, and quality diagnostics

Each video uses a fresh model and ByteTrack instance. Per-frame outputs contain `source_video`, frame/time, camera-local track ID, bbox, center, confidence, and normalized spatial coordinates for Notebook 15. Overlay trajectory tails are also reset between videos.

Fragmentation, gaps, boundary proximity, trajectory-step jitter, bbox-size jitter, short tracks, and lifespan are diagnostics only. Occlusion-related breaks, crossing-related ID changes, and biological identity cannot be determined automatically without TOP identity ground truth and visual review. Nothing here is HOTA, IDF1, an official ID switch count, or official MOT accuracy.


In [16]:
TRACKING_OUTPUT_DIR = PROJECT_ROOT / 'outputs/top/tracking'
TRACKING_RESULTS_PATH = PROJECT_ROOT / 'results/tracking/top_bytetrack_baseline_summary.csv'
TRACKING_LOG_DIR = PROJECT_ROOT / 'logs/tracking' / TRACK_EXPERIMENT_ID
expected_tracking_outputs = []
for context in TOP_VIDEO_CONTEXTS:
    stem = Path(context['source_video']).stem
    expected_tracking_outputs.extend([
        TRACKING_OUTPUT_DIR / f'{stem}_tracking_raw.csv',
        TRACKING_OUTPUT_DIR / f'{stem}_tracking_overlay.mp4',
    ])
existing_tracking_outputs = [path for path in [*expected_tracking_outputs, TRACKING_RESULTS_PATH] if path.exists()]
if TRACKING_LOG_DIR.exists() and any(TRACKING_LOG_DIR.iterdir()): existing_tracking_outputs.append(TRACKING_LOG_DIR)
if existing_tracking_outputs:
    raise RuntimeError(f'FAIL tracking preflight: preserve existing outputs before rerun: {[str(path.relative_to(PROJECT_ROOT)) for path in existing_tracking_outputs]}')

TRACKING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRACKING_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
TRACKING_LOG_DIR.mkdir(parents=True, exist_ok=False)
TRACKING_SUMMARIES = []
ACTUAL_TRACKER_CONFIGS = {}

for context in TOP_VIDEO_CONTEXTS:
    source_video = context['source_video']; stem = Path(source_video).stem
    raw_path = TRACKING_OUTPUT_DIR / f'{stem}_tracking_raw.csv'
    overlay_path = TRACKING_OUTPUT_DIR / f'{stem}_tracking_overlay.mp4'

    # Critical reset: a new model/tracker and new tail buffer for every source video.
    model = YOLO(str(BEST_MODEL), task='detect')
    trajectory_tails = defaultdict(lambda: deque(maxlen=50))
    capture = cv2.VideoCapture(str(context['path']))
    writer = cv2.VideoWriter(str(overlay_path), cv2.VideoWriter_fourcc(*'mp4v'), context['fps'], (context['width'], context['height']))
    assert capture.isOpened() and writer.isOpened(), f'FAIL tracking output setup for {source_video}'
    rows = []; frame_index = 0; started = time.perf_counter(); actual_config = None
    print(f'TRACKING START {source_video}: tracker state RESET; frames={context["frame_count"]}')
    try:
        while True:
            ok, frame = capture.read()
            if not ok: break
            result = model.track(
                source=frame, persist=True, tracker=str(TRACKER_CONFIG_PATH), conf=TRACK_DETECTION_FLOOR,
                iou=NMS_IOU, imgsz=IMGSZ, device=DEVICE, verbose=False,
            )[0]
            if actual_config is None:
                args = model.predictor.trackers[0].args
                actual_config = {key: getattr(args, key) for key in EXPECTED_TRACKER}
                assert actual_config == EXPECTED_TRACKER, f'FAIL actual tracker config for {source_video}: {actual_config}'
                ACTUAL_TRACKER_CONFIGS[source_video] = actual_config
                print(f'{source_video} actual tracker config: {actual_config}')

            boxes = result.boxes
            observed = boxes is not None and boxes.id is not None and len(boxes.id) > 0
            overlay = result.plot()
            if observed:
                track_ids = boxes.id.detach().cpu().numpy().astype(int)
                confidences = boxes.conf.detach().cpu().numpy().astype(float)
                xyxy = boxes.xyxy.detach().cpu().numpy().astype(float)
                for track_id, confidence, coords in zip(track_ids, confidences, xyxy):
                    x1, y1, x2, y2 = coords.tolist(); cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                    cx_norm, cy_norm = cx / context['width'], cy / context['height']
                    width_norm, height_norm = (x2 - x1) / context['width'], (y2 - y1) / context['height']
                    boundary_near = bool(x1 <= 0.01 * context['width'] or y1 <= 0.01 * context['height'] or x2 >= 0.99 * context['width'] or y2 >= 0.99 * context['height'])
                    rows.append({
                        'source_video': source_video, 'video_id': context['video_id'], 'frame_index': frame_index,
                        'time_sec': frame_index / context['fps'], 'track_id': int(track_id), 'x1': x1, 'y1': y1,
                        'x2': x2, 'y2': y2, 'cx': cx, 'cy': cy, 'confidence': float(confidence),
                        'cx_norm': cx_norm, 'cy_norm': cy_norm, 'bbox_width_norm': width_norm,
                        'bbox_height_norm': height_norm, 'boundary_near': boundary_near,
                    })
                    trajectory_tails[int(track_id)].append((int(round(cx)), int(round(cy))))
                for track_id, points in trajectory_tails.items():
                    if len(points) >= 2:
                        cv2.polylines(overlay, [np.asarray(points, dtype=np.int32)], False, (0, 255, 255), 2, cv2.LINE_AA)
            else:
                rows.append({
                    'source_video': source_video, 'video_id': context['video_id'], 'frame_index': frame_index,
                    'time_sec': frame_index / context['fps'], 'track_id': np.nan, 'x1': np.nan, 'y1': np.nan,
                    'x2': np.nan, 'y2': np.nan, 'cx': np.nan, 'cy': np.nan, 'confidence': np.nan,
                    'cx_norm': np.nan, 'cy_norm': np.nan, 'bbox_width_norm': np.nan,
                    'bbox_height_norm': np.nan, 'boundary_near': False,
                })
            writer.write(overlay); frame_index += 1
            if frame_index % PROGRESS_INTERVAL == 0 or frame_index == context['frame_count']:
                elapsed = time.perf_counter() - started
                print(f'{source_video}: {frame_index}/{context["frame_count"]} | elapsed={elapsed:.1f}s | processing FPS={frame_index/elapsed:.2f}')
    finally:
        capture.release(); writer.release()
    assert frame_index == context['frame_count'], f'FAIL tracking: incomplete {source_video}: {frame_index}/{context["frame_count"]}'

    runtime_sec = time.perf_counter() - started
    tracking_df = pd.DataFrame(rows)
    tracking_df.to_csv(raw_path, index=False)
    observed_df = tracking_df[tracking_df.track_id.notna()].copy()
    observed_df['track_id'] = observed_df.track_id.astype(int)
    active_counts = observed_df.groupby('frame_index').size().reindex(range(context['frame_count']), fill_value=0).astype(int)

    lifespan_rows = []; step_jitter_values = []; box_jitter_values = []
    for track_id, group in observed_df.groupby('track_id'):
        group = group.sort_values('frame_index'); frames = group.frame_index.to_numpy(dtype=int)
        frame_diffs = np.diff(frames); gap_mask = frame_diffs > 1
        consecutive = np.r_[False, frame_diffs == 1]
        center_step = np.sqrt(group.cx_norm.diff() ** 2 + group.cy_norm.diff() ** 2)
        box_step = np.sqrt(group.bbox_width_norm.diff() ** 2 + group.bbox_height_norm.diff() ** 2)
        step_jitter_values.extend(center_step[consecutive].dropna().tolist())
        box_jitter_values.extend(box_step[consecutive].dropna().tolist())
        lifespan_rows.append({
            'track_id': int(track_id), 'first_frame': int(frames.min()), 'last_frame': int(frames.max()),
            'observed_frames': len(frames), 'observed_lifespan_sec': len(frames) / context['fps'],
            'nominal_span_sec': (frames.max() - frames.min() + 1) / context['fps'],
            'gap_events': int(gap_mask.sum()), 'missing_frames_inside_span': int(np.maximum(frame_diffs - 1, 0).sum()),
            'long_gap_events_ge_1s': int((frame_diffs > context['fps']).sum()),
            'track_segments': int(gap_mask.sum()) + 1, 'boundary_near_rate': float(group.boundary_near.mean()),
        })
    lifespan_df = pd.DataFrame(lifespan_rows)
    detection_summary = next(row for row in VIDEO_DETECTION_SUMMARIES if row['video'] == source_video)
    max_lifespan = float(lifespan_df.observed_lifespan_sec.max()) if len(lifespan_df) else 0.0
    trajectory_usable = bool(len(observed_df) and max_lifespan >= 5.0)
    per_video_decision = 'PASS_WITH_WARNING' if trajectory_usable else 'HOLD'
    TRACKING_SUMMARIES.append({
        'video': source_video, 'video_id': context['video_id'], 'fps': context['fps'],
        'frame_count': context['frame_count'], 'duration_sec': context['duration_sec'],
        'mean_detections_per_frame': detection_summary['mean_detections_per_frame'],
        'median_detections_per_frame': detection_summary['median_detections_per_frame'],
        'mean_tracks_per_frame': float(active_counts.mean()), 'median_tracks_per_frame': float(active_counts.median()),
        'unique_track_ids': int(observed_df.track_id.nunique()),
        'median_lifespan_sec': float(lifespan_df.observed_lifespan_sec.median()) if len(lifespan_df) else 0.0,
        'mean_lifespan_sec': float(lifespan_df.observed_lifespan_sec.mean()) if len(lifespan_df) else 0.0,
        'max_lifespan_sec': max_lifespan,
        'short_tracks_le_1s': int((lifespan_df.observed_lifespan_sec <= 1).sum()) if len(lifespan_df) else 0,
        'short_tracks_le_2s': int((lifespan_df.observed_lifespan_sec <= 2).sum()) if len(lifespan_df) else 0,
        'tracks_ge_5s': int((lifespan_df.observed_lifespan_sec >= 5).sum()) if len(lifespan_df) else 0,
        'tracks_ge_10s': int((lifespan_df.observed_lifespan_sec >= 10).sum()) if len(lifespan_df) else 0,
        'gap_events': int(lifespan_df.gap_events.sum()) if len(lifespan_df) else 0,
        'fragmented_tracks_with_gaps': int((lifespan_df.gap_events > 0).sum()) if len(lifespan_df) else 0,
        'missing_frames_inside_spans': int(lifespan_df.missing_frames_inside_span.sum()) if len(lifespan_df) else 0,
        'long_gap_events_ge_1s': int(lifespan_df.long_gap_events_ge_1s.sum()) if len(lifespan_df) else 0,
        'boundary_near_box_rate': float(observed_df.boundary_near.mean()) if len(observed_df) else 0.0,
        'median_center_step_norm': float(np.median(step_jitter_values)) if step_jitter_values else None,
        'p95_center_step_norm': float(np.quantile(step_jitter_values, 0.95)) if step_jitter_values else None,
        'median_box_size_step_norm': float(np.median(box_jitter_values)) if box_jitter_values else None,
        'p95_box_size_step_norm': float(np.quantile(box_jitter_values, 0.95)) if box_jitter_values else None,
        'processing_fps': context['frame_count'] / runtime_sec,
        'trajectory_usable_for_spatial_features': trajectory_usable, 'decision': per_video_decision,
        'raw_output': str(raw_path.relative_to(PROJECT_ROOT)), 'overlay_output': str(overlay_path.relative_to(PROJECT_ROOT)),
    })
    print(f'Created {raw_path.relative_to(PROJECT_ROOT)} ({len(tracking_df)} rows)')
    print(f'Created {overlay_path.relative_to(PROJECT_ROOT)} ({overlay_path.stat().st_size} bytes)')

TRACKING_SUMMARY_DF = pd.DataFrame(TRACKING_SUMMARIES)
TRACKING_SUMMARY_DF.to_csv(TRACKING_RESULTS_PATH, index=False)
TRACKING_OVERALL_DECISION = 'HOLD' if (TRACKING_SUMMARY_DF.decision == 'HOLD').any() else 'PASS_WITH_WARNING'
TRAJECTORY_READY_FOR_NOTEBOOK_15 = bool(TRACKING_SUMMARY_DF.trajectory_usable_for_spatial_features.all())
TRACKING_WARNINGS = [
    'True fish count is unavailable; exact-count accuracy, count MAE, and proliferation factor are not reported.',
    'Track IDs are local to each video and are not biological or cross-camera identities.',
    'Occlusion-related breaks and crossing-related ID changes require visual/identity-ground-truth review.',
    'Reported gaps, segments, jitter, and lifespans are diagnostics, not HOTA, IDF1, official ID switches, or official MOT accuracy.',
]
(TRACKING_LOG_DIR / 'config.yaml').write_text(yaml.safe_dump({
    'experiment_id': TRACK_EXPERIMENT_ID, 'model': str(BEST_MODEL.relative_to(PROJECT_ROOT)),
    'model_sha256': BEST_MODEL_SHA256, 'imgsz': IMGSZ, 'tracking_detection_floor': TRACK_DETECTION_FLOOR,
    'iou': NMS_IOU, 'requested_tracker_config': EXPECTED_TRACKER,
    'actual_tracker_configs': ACTUAL_TRACKER_CONFIGS, 'videos': TOP_VIDEO_SPECS,
    'true_fish_count': None, 'tracker_state_reset_per_video': True,
}, sort_keys=False), encoding='utf-8')
(TRACKING_LOG_DIR / 'environment.txt').write_text((DETECTION_LOG_DIR / 'environment.txt').read_text(encoding='utf-8'), encoding='utf-8')
(TRACKING_LOG_DIR / 'summary.json').write_text(json.dumps({
    'experiment_id': TRACK_EXPERIMENT_ID,
    'per_video': {row['video']: row for row in TRACKING_SUMMARIES},
    'overall_interpretation': {
        'decision': TRACKING_OVERALL_DECISION,
        'trajectory_ready_for_notebook_15': TRAJECTORY_READY_FOR_NOTEBOOK_15,
        'criterion': 'Each video must contain at least one observed track with >=5 seconds of observed lifespan; visual review remains required.',
        'tracker_state_reset_per_video': True,
    },
    'metric_scope': 'TOP tracking diagnostics without identity ground truth',
    'warnings': TRACKING_WARNINGS,
}, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
display(TRACKING_SUMMARY_DF)
print(f'Created {TRACKING_RESULTS_PATH.relative_to(PROJECT_ROOT)} ({len(TRACKING_SUMMARY_DF)} rows)')
print(f'TOP TRACKING DECISION: {TRACKING_OVERALL_DECISION}')
print('These are tracking diagnostics, not official MOT identity metrics.')


TRACKING START 1.mp4: tracker state RESET; frames=1119
1.mp4 actual tracker config: {'tracker_type': 'bytetrack', 'track_high_thresh': 0.68, 'track_low_thresh': 0.5, 'new_track_thresh': 0.68, 'track_buffer': 15, 'match_thresh': 0.8, 'fuse_score': True}
1.mp4: 200/1119 | elapsed=4.6s | processing FPS=43.17
1.mp4: 400/1119 | elapsed=9.5s | processing FPS=42.16
1.mp4: 600/1119 | elapsed=14.1s | processing FPS=42.57
1.mp4: 800/1119 | elapsed=18.8s | processing FPS=42.47
1.mp4: 1000/1119 | elapsed=23.6s | processing FPS=42.39
1.mp4: 1119/1119 | elapsed=26.4s | processing FPS=42.40
Created outputs/top/tracking/1_tracking_raw.csv (2424 rows)
Created outputs/top/tracking/1_tracking_overlay.mp4 (37291336 bytes)
TRACKING START 2.mp4: tracker state RESET; frames=2031
2.mp4 actual tracker config: {'tracker_type': 'bytetrack', 'track_high_thresh': 0.68, 'track_low_thresh': 0.5, 'new_track_thresh': 0.68, 'track_buffer': 15, 'match_thresh': 0.8, 'fuse_score': True}
2.mp4: 200/2031 | elapsed=4.6s | pr

,video,video_id,fps,frame_count,duration_sec,mean_detections_per_frame,median_detections_per_frame,mean_tracks_per_frame,median_tracks_per_frame,unique_track_ids,...,boundary_near_box_rate,median_center_step_norm,p95_center_step_norm,median_box_size_step_norm,p95_box_size_step_norm,processing_fps,trajectory_usable_for_spatial_features,decision,raw_output,overlay_output
0,1.mp4,TOP_VIDEO_1,19.350643,1119,57.827536,1.887399,2.0,2.159964,2.0,33,...,0.002482,0.002028,0.010228,0.002635,0.009789,42.396589,True,PASS_WITH_WARNING,outputs/top/tracking/1_tracking_raw.csv,outputs/top/tracking/1_tracking_overlay.mp4
1,2.mp4,TOP_VIDEO_2,17.172190,2031,118.272623,2.910881,3.0,3.116199,3.0,53,...,0.000000,0.002051,0.010487,0.002590,0.009535,41.903804,True,PASS_WITH_WARNING,outputs/top/tracking/2_tracking_raw.csv,outputs/top/tracking/2_tracking_overlay.mp4


Created results/tracking/top_bytetrack_baseline_summary.csv (2 rows)
TOP TRACKING DECISION: PASS_WITH_WARNING
These are tracking diagnostics, not official MOT identity metrics.


## 17. Output paths
## 18. Warnings and scientific limitations
## 19. Summary by video
## 20. Final decision
## 21. Next step

The detector is already accepted and is not retrained. The final tracking decision is based on two independent videos and remains conservative because TOP identity ground truth and a verified true fish count are unavailable. Notebook 15 is not started automatically.


In [17]:
DETECTOR_DECISION = 'PASS'
NOTEBOOK_14_DECISION = 'HOLD' if TRACKING_OVERALL_DECISION == 'HOLD' else 'PASS_WITH_WARNING'
FINAL_SUMMARY = {
    'status': NOTEBOOK_14_DECISION,
    'video_section_runtime_sec': round(time.perf_counter() - VIDEO_SECTION_START_TIME, 3),
    'top_detector': {
        'decision': DETECTOR_DECISION, 'model': str(BEST_MODEL.relative_to(PROJECT_ROOT)),
        'model_sha256': BEST_MODEL_SHA256, 'best_epoch': int(ACCEPTED_TRAIN_METRICS['best_epoch']),
        'precision': float(ACCEPTED_VALIDATION_METRICS['precision']),
        'recall': float(ACCEPTED_VALIDATION_METRICS['recall']),
        'mAP50': float(ACCEPTED_VALIDATION_METRICS['mAP50']),
        'mAP50_95': float(ACCEPTED_VALIDATION_METRICS['mAP50_95']),
        'training_rerun': False, 'validation_rerun': False,
    },
    'top_tracking': {
        'decision': TRACKING_OVERALL_DECISION,
        'per_video': {row['video']: row for row in TRACKING_SUMMARIES},
        'trajectory_ready_for_notebook_15': TRAJECTORY_READY_FOR_NOTEBOOK_15,
    },
    'warnings': TRACKING_WARNINGS,
    'next_step': 'USER reviews both overlays and metrics. Do not start Notebook 15 without explicit approval.',
}
print('FINAL SUMMARY')
print(json.dumps(FINAL_SUMMARY, indent=2, ensure_ascii=False, default=str))
print('OUTPUT PATHS')
for path in [
    PROJECT_ROOT / 'outputs/top/detection/1_detection_per_frame.csv',
    PROJECT_ROOT / 'outputs/top/detection/2_detection_per_frame.csv',
    PROJECT_ROOT / 'outputs/top/tracking/1_tracking_raw.csv',
    PROJECT_ROOT / 'outputs/top/tracking/2_tracking_raw.csv',
    PROJECT_ROOT / 'outputs/top/tracking/1_tracking_overlay.mp4',
    PROJECT_ROOT / 'outputs/top/tracking/2_tracking_overlay.mp4',
    DETECTION_RESULTS_PATH, TRACKING_RESULTS_PATH,
    TRACKING_LOG_DIR / 'summary.json',
]:
    print(f'{path.relative_to(PROJECT_ROOT)} | exists={path.exists()} | size={path.stat().st_size if path.exists() else None}')
print(f'TOP DETECTOR: {DETECTOR_DECISION}')
print(f'TOP TRACKING: {TRACKING_OVERALL_DECISION}')
print(f'NOTEBOOK 14: {NOTEBOOK_14_DECISION}')
print(f'Trajectory ready for Notebook 15: {TRAJECTORY_READY_FOR_NOTEBOOK_15}')
print('STOP — USER review is required before Notebook 15.')


FINAL SUMMARY
{
  "status": "PASS_WITH_WARNING",
  "video_section_runtime_sec": 507.483,
  "top_detector": {
    "decision": "PASS",
    "model": "runs/top/yolov8n_top_v2_baseline/weights/best.pt",
    "model_sha256": "216174c3a40d57bfd7b0d7e46eec90974ddb0b27c879dd1961543e3233a6c5e5",
    "best_epoch": 65,
    "precision": 0.967530547941785,
    "recall": 0.9713393817483136,
    "mAP50": 0.9834169324420174,
    "mAP50_95": 0.6062227869655318,
    "training_rerun": false,
    "validation_rerun": false
  },
  "top_tracking": {
    "decision": "PASS_WITH_WARNING",
    "per_video": {
      "1.mp4": {
        "video": "1.mp4",
        "video_id": "TOP_VIDEO_1",
        "fps": 19.350642974397303,
        "frame_count": 1119,
        "duration_sec": 57.827535833333336,
        "mean_detections_per_frame": 1.8873994638069704,
        "median_detections_per_frame": 2.0,
        "mean_tracks_per_frame": 2.159964253798034,
        "median_tracks_per_frame": 2.0,
        "unique_track_ids": 33,
  